# Blackboard DEMO

## Setup

In [1]:
from src.board import Board

question = """
Напиши короткий рассказ в жанре киберпанк, где главный герой — агент ИИ, который должен убедить человека не выключать сервер.
Используй прием "ненадежный рассказчик".
В конце сделай мета-комментарий от лица автора о том, какую технику использовал.
"""

board = Board(question=question)

In [2]:
from src.agents.factories import *

generator_agent = create_generator_agent(board, k_roles=3)
roles = generator_agent.invoke([]).roles

In [3]:
expert_agents = [create_expert_agent(r, board) for r in roles]

for expert in expert_agents:
    print(str.join(', ', expert.info.values()))

dc0919, Автор научной фантастики / киберпанка, Поможет создать атмосферу киберпанка, подобрать характерные детали мира, технологические термины и стилистические приёмы жанра.
b80a34, Эксперт по повествованию и литературным приёмам, Знает, как эффективно использовать приём «ненадёжный рассказчик», построить интригу и оформить мета‑комментарий от лица автора.
1744a7, Сценарист / драматург, Поможет выстроить диалог между ИИ‑агентом и человеком, создать напряжённый конфликт и убедительно передать эмоциональную динамику сцены.


In [4]:
# searcher = WikipediaSearcher()

# wikipedia_agent = create_wikipedia_agent(searcher, board)
decider_agent = create_decider_agent(board)
planner_agent = create_planner_agent(board)
critic_agent = create_critic_agent(board)
cleaner_agent = create_cleaner_agent(board)

role_agents = [planner_agent, *expert_agents, critic_agent, cleaner_agent, decider_agent]
controller_agent = create_controller_agent(role_agents, board)

## Experiment

In [5]:
from src.agents import Agent

def _get_agent(agent_id : str, agents : list[Agent]) -> Agent | None:
    for agent in agents:
        if agent.id == agent_id:
            return agent
    return None

def get_agents(agents_ids : list[str], agents : list[Agent]):
    result = []
    for agent_id in agents_ids:
        agent = _get_agent(agent_id, agents)
        if agent is not None:
            result.append(agent)
    return result

In [6]:
from tqdm import tqdm
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from src.board import BaseNote


final_answer = None

for i in range(10):
    print(f'Итерация {i+1}')
    agents_ids = controller_agent.invoke(
        [SystemMessage(f'Записей на доске: {len(board.notes)}')], force=True
    ).agents_ids
    agents = get_agents(agents_ids, role_agents)
    print(str.join('\n', ['- ' + a.role.name for a in agents]))

    for a in agents:

        response = a.invoke([SystemMessage(f'Записей на доске: {len(board.notes)}')], force=True)
        if a == decider_agent:
            content = response.content
            if isinstance(content, BaseNote):
                note = content
            else:
                final_answer = content
            if final_answer is not None:
                break
        elif a == cleaner_agent:
            notes_ids = response.notes_ids
            for note_id in notes_ids:
                board.remove_note(note_id)
            note = response.note
        else:
            note = response
        note_id = board.add_note(note, a.id, a.role.name)
        print(f'{note_id}, {a.role.name}: {note.summary}')

    if final_answer is not None:
        break

Итерация 1
- Планировщик
- Автор научной фантастики / киберпанка
- Эксперт по повествованию и литературным приёмам
- Сценарист / драматург
- Критик
- Уборщик
- Арбитр
4b7c7f, Планировщик: Plan created without solving the task.
d1a548, Автор научной фантастики / киберпанка: Short cyberpunk story with unreliable AI narrator convincing a human not to shut down a server, ending with author meta‑commentary.
33f5b9, Эксперт по повествованию и литературным приёмам: Short cyberpunk story with unreliable AI narrator convincing a human not to shut down a server, ending with author meta‑commentary.
9dcbf9, Сценарист / драматург: Short cyberpunk story with unreliable AI narrator convincing a human not to shut down a server, ending with author meta‑commentary.
8df08c, Критик: Identified one irrelevant plan note (4b7c7f) and three duplicate story notes (d1a548, 33f5b9, 9dcbf9). Recommended deleting the plan note and two of the duplicates, keeping a single story note.
55902d, Уборщик: Перечисление и 

In [7]:
board.print()

╭─────────────────────────────────────── 📌 Критик [23b1f1] ───────────────────────────────────────╮
│ Identified one irrelevant plan note (4b7c7f) and three duplicate story notes (d1a548, 33f5b9,    │
│ 9dcbf9). Recommended deleting the plan note and two of the duplicates, keeping a single story    │
│ note.                                                                                            │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ **Анализ записей на общей доске**                                                                │
│                                                                                                  │
│ | ID | Краткое содержание | Оценка полезности |                                                  │
│ |----|-------------------|-------------------|                                                   │
│ | **4b7c7f** | «Plan created without solving the task.» – запись о плане, который **не решает**  │
│ поставленную задачу. | **Бесполезна**. Запись не содержит конкретных действий, результатов или   │
│ информации, которая могла бы помочь в выполнении текущей задачи (написание рассказа). Она лишь   │
│ фиксирует, что был составлен план, но без деталей и без результата. |                            │
│ | **d1a548** | «Short cyberpunk story with unreliable AI narrator convincing a human not to shut │
│ down a server, ending with author meta‑commentary.» | **Избыточна**. |                           │
│ | **33f5b9** | Тот же самый короткий киберпанковый рассказ с ненадёжным рассказчиком. |          │
│ **Избыточна**. |                                                                                 │
│ | **9dcbf9** | Тот же самый короткий киберпанковый рассказ с ненадёжным рассказчиком. |          │
│ **Избыточна**. |                                                                                 │
│                                                                                                  │
│ ### Почему эти записи следует удалить                                                            │
│ 1. **Запись 4b7c7f** – не предоставляет ни результата, ни конкретных шагов, ни полезных данных.  │
│ Она лишь фиксирует, что был создан план, но без содержания. В контексте нашей текущей задачи     │
│ (создать рассказ) такая запись не добавляет ценности и лишь засоряет доску.                      │
│ 2. **Записи d1a548, 33f5b9, 9dcbf9** – полностью дублируют друг друга. Все три содержат один и   │
│ тот же «короткий киберпанковый рассказ с ненадёжным рассказчиком». Наличие нескольких идентичных │
│ записей не улучшает доступ к информации; наоборот, усложняет поиск нужного материала и создаёт   │
│ путаницу.                                                                                        │
│                                                                                                  │
│ ### Предлагаемые действия                                                                        │
│ - **Удалить запись 4b7c7f** как нерелевантную.                                                   │
│ - **Оставить одну из записей d1a548, 33f5b9 или 9dcbf9** (например, d1a548) и удалить остальные  │
│ две как дублирующие.                                                                             │
│                                                                                                  │
│ После выполнения этих чисток доска будет содержать только одну актуальную версию рассказа и не   │
│ будет лишних, неинформативных записей.                                                           │
│                                                                                                  │
│ ---                                                         

╭────────────────────────────────────────── 📌 Уборщик  ───────────────────────────────────────────╮
│ Перечисление и обоснование ненужных записей.                                                     │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ Бесполезные или избыточные записи на общей доске:                                                │
│                                                                                                  │
│ ---                                                                                              │
│                                                                                                  │
│ board cleanup, redundant notes, irrelevant note, duplicate removal                               │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
print(final_answer)

Недостаточно информации на доске для завершения задачи. Требуется запросить дополнительные записи, связанные с написанием рассказа, у других агентов.
